<a href="https://colab.research.google.com/github/bruno-ritter/brazilain_stock_volatility_and_google_trends/blob/main/POC_TCC_Google_Trends_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarefas

*  Limpar a base antes de rodar os modelos: retirar nan, retirar svi não-estacionários
*  Fazer o modelo multivariado GARCH
* Comparar os três modelos









# Bibliotecas

In [1]:
!pip install yfinance pandas pytrends tqdm arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 19.0 MB/s eta 0:00:00


In [2]:
import time
import math
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import arch

from pytrends.request import TrendReq
from datetime import datetime, timedelta

from arch import arch_model
from arch.univariate import ConstantMean, GARCH, Normal


In [3]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

# Coleta Google Trends v3: Ticker

Para cada ação, coleta o SVI um termo:
  1. TICKER     — ex: "PETR4"

In [ ]:
# ──────────────────────────────────────────────────────────────
# CONFIGURAÇÕES
# ──────────────────────────────────────────────────────────────
ANOS_HISTORICO    = 5
VOLUME_MIN_DIARIO = 1
DATA_INICIO       = (datetime.today() - timedelta(days=365 * ANOS_HISTORICO)).strftime("%Y-%m-%d")
DATA_FIM          = datetime.today().strftime("%Y-%m-%d")
PAUSA_TRENDS      = 8     # segundos entre requisições

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 1 — PREÇOS
# ──────────────────────────────────────────────────────────────

# Todos os ticker
tickers = pd.read_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/acoes-listadas-b3.csv")
TERMOS = list(tickers.Ticker)
tickers_yf = [t + ".SA" for t in TERMOS]

dados_raw = yf.download(
    tickers_yf, start=DATA_INICIO, end=DATA_FIM,
    auto_adjust=True, progress=True,
)

registros_acoes = []
for ticker_yf in tickers_yf:

    try:
        close  = dados_raw["Close"][ticker_yf].dropna()
        volume = dados_raw["Volume"][ticker_yf].dropna()

        if len(close) < ANOS_HISTORICO * 252 * 0.85:
            print(f"  [SKIP] {ticker_yf} — histórico insuficiente ({len(close)} dias)")
            continue

        vol_fin = (volume * close.mean()).mean()
        if vol_fin < VOLUME_MIN_DIARIO:
            print(f"  [SKIP] {ticker_yf} — liquidez insuficiente")
            continue

        registros_acoes.append({
            "ticker_yf": ticker_yf,
            "dias": len(close),
            "vol_fin_medio": round(vol_fin, 2),
        })
        print(f"  [OK] {ticker_yf} | Vol R${vol_fin:>14,.0f}")

    except Exception as e:
        print(f"  [ERRO] {ticker_yf}: {e}")

df_acoes = pd.DataFrame(registros_acoes)
df_acoes.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/acoes_elegiveis_v3.csv", index=False)
print(f"\n✅ {len(df_acoes)} ações elegíveis → acoes_elegiveis_v3.csv")

[*********************100%***********************]  387 of 387 completed
ERROR:yfinance:
3 Failed downloads:
ERROR:yfinance:['PINE11.SA', 'AZEV11.SA', 'BIOM11.SA']: YFTzMissingError('possibly delisted; no timezone found')


  [OK] PETR4.SA | Vol R$ 1,217,024,729
  [OK] B3SA3.SA | Vol R$   501,721,801
  [OK] HAPV3.SA | Vol R$   414,914,099
  [OK] BEEF3.SA | Vol R$    81,939,864
  [OK] GMAT3.SA | Vol R$    35,363,394
  [OK] ENEV3.SA | Vol R$   124,340,396
  [OK] CSAN3.SA | Vol R$   202,521,541
  [OK] RAIZ4.SA | Vol R$    62,220,525
  [OK] COGN3.SA | Vol R$    87,976,168
  [OK] MBRF3.SA | Vol R$   115,069,877
  [OK] BBDC4.SA | Vol R$   561,353,428
  [OK] VAMO3.SA | Vol R$    63,215,193
  [OK] ITUB4.SA | Vol R$   773,960,443
  [OK] ABEV3.SA | Vol R$   315,428,296
  [OK] CPLE3.SA | Vol R$    35,167,495
  [OK] BBAS3.SA | Vol R$   491,285,576
  [OK] ITSA4.SA | Vol R$   208,846,388
  [OK] CSNA3.SA | Vol R$   138,195,652
  [OK] VALE3.SA | Vol R$ 1,487,690,418
  [OK] PRIO3.SA | Vol R$   469,340,598
  [OK] MGLU3.SA | Vol R$   722,088,461
  [OK] PETR3.SA | Vol R$   344,994,118
  [OK] CMIG4.SA | Vol R$    99,829,959
  [OK] ONCO3.SA | Vol R$    23,893,817
  [OK] CVCB3.SA | Vol R$   107,887,469
  [SKIP] NATU3.SA — histó

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 2 — VOLATILIDADE REALIZADA
# ──────────────────────────────────────────────────────────────

series_semanais = []
for _, row in df_acoes.iterrows():
    try:
        close  = dados_raw["Close"][row["ticker_yf"]].dropna()
        volume = dados_raw["Volume"][row["ticker_yf"]].dropna()

        df_d = pd.DataFrame({"close": close, "volume": volume})
        df_d.index = pd.to_datetime(df_d.index)

        df_d["log_ret"] = np.log(df_d["close"] / df_d["close"].shift(1))

        # Agrega semanalmente (semana fecha na sexta)
        wk = df_d.resample("W-FRI").agg(
            preco_abertura   = ("close",   "first"),
            preco_fechamento = ("close",   "last"),
            volume_total     = ("volume",  "sum"),
            n_preços         = ("log_ret", "count"),      # nº de pregões na semana
            rv               = ("log_ret", lambda x: (x**2).sum()),  # RV = Σ r²
        ).dropna(subset=["rv"])

        wk["ticker_yf"] = row["ticker_yf"]
        wk["ticker_b3"] = row["ticker_yf"].replace(".SA", "")
        series_semanais.append(wk.reset_index().rename(columns={"Date": "semana"}))

    except Exception as e:
        print(f"  [ERRO preços {row['ticker_yf']}]: {e}")

df_precos = pd.concat(series_semanais, ignore_index=True)
df_precos.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/precos_semanais_v3.csv", index=False)
print(f"✅ {len(df_precos)} linhas → precos_semanais_v3.csv")

✅ 93822 linhas → precos_semanais_v3.csv


In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 3 — GOOGLE TRENDS: SVI POR TICKER
# ──────────────────────────────────────────────────────────────

pytrends = TrendReq(hl="pt-BR", tz=-180)

def coletar_svi(ticker):
    """Retorna DataFrame semanal do SVI para um ticker ou None."""
    try:
        pytrends.build_payload(
            kw_list=[ticker],
            timeframe=f"{DATA_INICIO} {DATA_FIM}",
            geo="BR",
            gprop="",
        )
        df = pytrends.interest_over_time()
        if df.empty:
            return None
        df = df.drop(columns=["isPartial"], errors="ignore")
        df.columns = ["svi"]
        df.index.name = "semana"
        return df.reset_index()
    except Exception as e:
        print(f"    [ERRO '{ticker}']: {e}")
        return None


todos_trends = []
log_coleta   = []
tickers_validos = list(df_precos["ticker_b3"].unique())

for ticker in tickers_validos:

    print(f"\n  {ticker}")
    df_t = coletar_svi(ticker)
    time.sleep(PAUSA_TRENDS)

    if df_t is None:
        print("VAZIO")
        log_coleta.append({
            "ticker": ticker, "status": "vazio",
            "semanas": 0, "svi_media": None, "svi_std": None,
        })
        continue

    svi_media = round(df_t["svi"].mean(), 2)
    svi_std   = round(df_t["svi"].std(), 2)
    svi_zeros = (df_t["svi"] == 0).mean()
    print(f"OK | {len(df_t)} sem | média={svi_media:.1f} "
          f"std={svi_std:.1f} zeros={svi_zeros:.0%}")

    for _, tr in df_t.iterrows():
        todos_trends.append({
            "semana": tr["semana"],
            "ticker_b3": ticker,
            "svi":    tr["svi"],
        })

    log_coleta.append({
        "ticker":       ticker,
        "status":       "ok",
        "semanas":      len(df_t),
        "svi_media":    svi_media,
        "svi_std":      svi_std,
        "svi_zeros_pct": round(svi_zeros * 100, 1),
    })


df_trends = pd.DataFrame(todos_trends)
df_log    = pd.DataFrame(log_coleta)

df_trends.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/google_trends_svi.csv", index=False)
df_log.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/log_coleta_svi.csv", index=False)

print(f"\n✅ {len(df_trends)} linhas → google_trends_svi.csv")
print(f"✅ Log de coleta    → log_coleta_svi.csv")


  PETR4
OK | 261 sem | média=29.3 std=9.8 zeros=0%

  B3SA3
OK | 261 sem | média=19.9 std=10.9 zeros=0%

  HAPV3
OK | 261 sem | média=10.8 std=9.2 zeros=0%

  BEEF3
OK | 261 sem | média=26.3 std=12.4 zeros=0%

  GMAT3
OK | 261 sem | média=25.1 std=14.7 zeros=0%

  ENEV3
OK | 261 sem | média=14.6 std=11.7 zeros=4%

  CSAN3
OK | 261 sem | média=18.6 std=9.4 zeros=0%

  RAIZ4
OK | 261 sem | média=21.3 std=11.6 zeros=6%

  COGN3
OK | 261 sem | média=34.4 std=15.6 zeros=0%

  MBRF3
OK | 261 sem | média=4.4 std=14.5 zeros=89%

  BBDC4
OK | 261 sem | média=24.1 std=9.1 zeros=0%

  VAMO3
OK | 261 sem | média=29.5 std=19.8 zeros=0%

  ITUB4
OK | 261 sem | média=40.5 std=11.1 zeros=0%

  ABEV3
OK | 261 sem | média=11.9 std=7.2 zeros=0%

  CPLE3
OK | 261 sem | média=39.9 std=19.0 zeros=0%

  BBAS3
OK | 261 sem | média=20.9 std=13.6 zeros=0%

  ITSA4
OK | 261 sem | média=42.7 std=13.5 zeros=0%

  CSNA3
OK | 261 sem | média=47.1 std=13.5 zeros=0%

  VALE3
OK | 261 sem | média=41.0 std=11.0 zeros=0

In [ ]:
df_trends = pd.read_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/google_trends_svi.csv")

,semana,ticker,svi
0,2021-03-21,PETR4,35
1,2021-03-28,PETR4,29
2,2021-04-04,PETR4,33
3,2021-04-11,PETR4,32
4,2021-04-18,PETR4,32
...,...,...,...
92128,2026-02-15,MERC4,0
92129,2026-02-22,MERC4,0
92130,2026-03-01,MERC4,0
92131,2026-03-08,MERC4,0


In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 4 — ESTACIONARIEDADE DO SVI (ADF POR TICKER)
# ──────────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller

NIVEL_SIGNIFICANCIA = 0.05

def teste_adf(serie, nome=""):
    """
    Roda o ADF e retorna dict com estatística, p-valor e conclusão.
    Usa seleção automática de lags (método 'AIC').
    """
    serie = serie.dropna()
    if len(serie) < 10:
        return {"stat": None, "pvalor": None, "estacionaria": None, "obs": "série curta"}

    stat, pvalor, _, _, valores_criticos, _ = adfuller(serie, autolag="AIC")
    estacionaria = pvalor < NIVEL_SIGNIFICANCIA

    print(f"      {'[OK]' if estacionaria else '[RAIZ UNITÁRIA]'} {nome}: "
          f"stat={stat:.3f}  p={pvalor:.4f}  {'estacionária' if estacionaria else 'não estacionária'}")

    return {
        "stat":         round(stat, 4),
        "pvalor":       round(pvalor, 4),
        "cv_1pct":      round(valores_criticos["1%"], 3),
        "cv_5pct":      round(valores_criticos["5%"], 3),
        "cv_10pct":     round(valores_criticos["10%"], 3),
        "estacionaria": estacionaria,
        "obs":          "",
    }


resultados_adf = []

for ticker, grupo in df_trends.groupby("ticker"):

    print(f"\n  {ticker}")
    svi = grupo.sort_values("semana")["svi"].reset_index(drop=True)

    # ── ADF no nível ──────────────────────────────────────────
    adf_nivel = teste_adf(svi, "SVI nível")

    # ── Primeira diferença e ADF ──────────────────────────────
    svi_diff  = svi.diff()
    adf_diff  = teste_adf(svi_diff, "SVI Δ(1)")

    resultados_adf.append({
        "ticker":                   ticker,
        # nível
        "svi_adf_stat":             adf_nivel["stat"],
        "svi_adf_pvalor":           adf_nivel["pvalor"],
        "svi_estacionario":         adf_nivel["estacionaria"],
        # primeira diferença
        "svi_diff_adf_stat":        adf_diff["stat"],
        "svi_diff_adf_pvalor":      adf_diff["pvalor"],
        "svi_diff_estacionario":    adf_diff["estacionaria"],
    })

    # ── Adiciona coluna svi_diff ao df_trends ─────────────────
    df_trends.loc[df_trends["ticker"] == ticker, "svi_diff"] = svi_diff.values


# ── Bases finais ──────────────────────────────────────────────
df_adf = pd.DataFrame(resultados_adf)

df_adf.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/log_adf_svi.csv", index=False)

# ── Resumo ────────────────────────────────────────────────────
n_nivel_ok = df_adf["svi_estacionario"].sum()
n_diff_ok  = df_adf["svi_diff_estacionario"].sum()
n          = len(df_adf)

print(f"\n{'─'*55}")
print(f"  SVI nível estacionário:      {n_nivel_ok}/{n} tickers")
print(f"  SVI Δ(1)  estacionário:      {n_diff_ok}/{n} tickers")
print(f"\n✅ google_trends_svi.csv  → svi + svi_diff por semana")
print(f"✅ log_adf_svi.csv        → resultado ADF por ticker")


  AALR3
      [OK] SVI nível: stat=-3.340  p=0.0132  estacionária
      [OK] SVI Δ(1): stat=-6.766  p=0.0000  estacionária

  ABCB4
      [RAIZ UNITÁRIA] SVI nível: stat=-2.829  p=0.0543  não estacionária
      [OK] SVI Δ(1): stat=-10.377  p=0.0000  estacionária

  ABEV3
      [OK] SVI nível: stat=-6.704  p=0.0000  estacionária
      [OK] SVI Δ(1): stat=-9.626  p=0.0000  estacionária

  AERI3
      [RAIZ UNITÁRIA] SVI nível: stat=-2.782  p=0.0608  não estacionária
      [OK] SVI Δ(1): stat=-11.900  p=0.0000  estacionária

  AGRO3
      [RAIZ UNITÁRIA] SVI nível: stat=-1.768  p=0.3966  não estacionária
      [OK] SVI Δ(1): stat=-12.253  p=0.0000  estacionária

  AGXY3
      [OK] SVI nível: stat=-2.887  p=0.0469  estacionária
      [OK] SVI Δ(1): stat=-8.392  p=0.0000  estacionária

  AHEB3
      [OK] SVI nível: stat=-16.125  p=0.0000  estacionária
      [OK] SVI Δ(1): stat=-8.468  p=0.0000  estacionária

  ALLD3
      [RAIZ UNITÁRIA] SVI nível: stat=-1.463  p=0.5518  não estacionária
 

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 5 — MERGE: PREÇOS + TRENDS
# Uma linha por (semana, ticker)
# ──────────────────────────────────────────────────────────────

df_trends["semana"] = pd.to_datetime(df_trends["semana"])
df_trends["ticker_b3"] = df_trends["ticker"]
df_precos["semana"] = pd.to_datetime(df_precos["semana"])

# Alinha domingo (Trends) → sexta (preços)
df_trends["semana"] = df_trends["semana"] + pd.offsets.Week(weekday=4)

df_final = pd.merge(
    df_precos,
    df_trends[["semana","ticker_b3","svi","svi_diff"]],
    on=["semana","ticker_b3"],
    how="inner",
).sort_values(["ticker","semana"]).reset_index(drop=True)

df_final.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/base_final_tcc.csv", index=False)

print(f"✅ BASE FINAL → base_final_tcc_v3.csv")

✅ BASE FINAL → base_final_tcc_v3.csv


# Especificação dos Modelos Competidores

In [ ]:
# ──────────────────────────────────────────────────────────────
# Selecionar apenas o ticker com SVI Diff estacionário
# ──────────────────────────────────────────────────────────────

df_final = pd.read_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/base_final_tcc.csv")
df_final["semana"] = pd.to_datetime(df_final["semana"])
df_adf = pd.read_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/log_adf_svi.csv")

#
df_final = df_final[df_final["ticker_b3"].isin(df_adf[df_adf['svi_diff_estacionario'] == True]["ticker"])]

df_final.head()

# MODELO 0 — PASSEIO ALEATÓRIO (Random Walk)
σ̂²_t = RV_{t-1}

In [ ]:
resultados_m0 = []

for ticker, grupo in df_final.groupby("ticker_b3"):

    df_t = (
        grupo
        .sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv"]]
        .copy()
    )

    # Previsão = RV da semana anterior
    df_t["rv_pred_m0"] = df_t["rv"].shift(1)

    # Remove semanas sem previsão (primeira observação de cada ticker)
    df_t = df_t.dropna(subset=["rv_pred_m0"])

    df_t["ticker_b3"] = ticker
    resultados_m0.append(df_t)


df_m0 = pd.concat(resultados_m0, ignore_index=True)
df_m0.to_csv("previsoes_m0.csv", index=False)

df_m0

,semana,rv,rv_pred_m0,ticker_b3
0,2021-04-16,0.000317,0.001092,AALR3
1,2021-04-23,0.000731,0.000317,AALR3
2,2021-04-30,0.000978,0.000731,AALR3
3,2021-05-07,0.001199,0.000978,AALR3
4,2021-05-14,0.016059,0.001199,AALR3
...,...,...,...,...
90982,2026-02-20,0.000838,0.001959,YDUQ3
90983,2026-02-27,0.005215,0.000838,YDUQ3
90984,2026-03-06,0.006552,0.005215,YDUQ3
90985,2026-03-13,0.026762,0.006552,YDUQ3


# MODELO 1: GARCH(1,1) PADRÃO
σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1}

Janela móvel de tamanho fixo: 52 * 3 semanas

In [ ]:
JANELA_MINIMA = 52 * 3   # tamanho fixo da janela (156 semanas)

resultados_m1 = []
log_m1        = []

for ticker, grupo in df_final.groupby("ticker_b3"):

    # ── 1. Prepara a série do ticker ──────────────────────────
    df_t = (
        grupo
        .sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv", "preco_abertura", "preco_fechamento"]]
        .copy()
    )

    # Retorno log semanal com média removida
    df_t["ret"] = np.log(df_t["preco_fechamento"] / df_t["preco_abertura"])
    df_t["ret"] = df_t["ret"] - df_t["ret"].mean()
    df_t = df_t.dropna(subset=["ret"]).reset_index(drop=True)

    n = len(df_t)
    if n < JANELA_MINIMA + 1:
        print(f"  [PULADO] {ticker}: {n} semanas (mínimo {JANELA_MINIMA + 1})")
        log_m1.append({"ticker_b3": ticker, "status": "série curta", "previsoes": 0})
        continue

    semana_corte = df_t["semana"].iloc[JANELA_MINIMA]
    print(f"  {ticker} — {n} sem | corte: {semana_corte.date()} "
          f"| previsões OOS: {n - JANELA_MINIMA}")

    previsoes = []

    # ── 2. Rolling window (janela fixa) ───────────────────────
    for t in range(JANELA_MINIMA, n):

        # Janela deslizante: sempre 156 semanas imediatamente anteriores
        retornos_treino = df_t["ret"].iloc[t - JANELA_MINIMA : t].values * 100

        try:
            # ── 3. Estima o GARCH(1,1) na janela ─────────────
            modelo = arch_model(
                retornos_treino,
                mean="Constant",
                vol="GARCH",
                p=1, q=1,
                dist="normal",
            )
            fit = modelo.fit(disp="off", show_warning=False)

            # ── 4. Previsão 1 passo à frente ──────────────────
            forecast = fit.forecast(horizon=1, reindex=False)
            var_pred = forecast.variance.values[-1, 0] / 10_000

            previsoes.append({
                "semana":      df_t["semana"].iloc[t],
                "rv":          df_t["rv"].iloc[t],
                "rv_pred_m1":  var_pred,
                "ticker_b3":   ticker,
                "omega":       fit.params.get("omega",    np.nan),
                "alpha":       fit.params.get("alpha[1]", np.nan),
                "beta":        fit.params.get("beta[1]",  np.nan),
            })

        except Exception as e:
            print(f"    [ERRO t={t} {ticker}]: {e}")
            previsoes.append({
                "semana":     df_t["semana"].iloc[t],
                "rv":         df_t["rv"].iloc[t],
                "rv_pred_m1": np.nan,
                "ticker_b3":  ticker,
                "omega": np.nan, "alpha": np.nan, "beta": np.nan,
            })

    df_ticker = pd.DataFrame(previsoes)
    df_ticker["split"] = np.where(
        df_ticker["semana"] >= semana_corte,
        "out-of-sample",
        "in-sample",
    )

    resultados_m1.append(df_ticker)
    log_m1.append({
        "ticker_b3":     ticker,
        "status":        "ok",
        "semana_corte":  semana_corte,
        "previsoes_oos": (df_ticker["split"] == "out-of-sample").sum(),
        "nan_pct":       round(df_ticker["rv_pred_m1"].isna().mean() * 100, 1),
    })


df_m1     = pd.concat(resultados_m1, ignore_index=True)
df_log_m1 = pd.DataFrame(log_m1)
df_m1.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/previsoes_m1.csv", index=False)

print(f"\n✅ {len(df_m1)} linhas → previsoes_m1.csv")
print(f"   in-sample:     {(df_m1['split'] == 'in-sample').sum()} observações")
print(f"   out-of-sample: {(df_m1['split'] == 'out-of-sample').sum()} observações")

  AALR3 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  ABCB4 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  ABEV3 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  AERI3 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  AGRO3 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  AGXY3 — 243 sem | corte: 2024-07-26 | previsões OOS: 87
  AHEB3 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  ALLD3 — 258 sem | corte: 2024-04-12 | previsões OOS: 102
  ALOS3 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  ALPA3 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  ALPA4 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  ALPK3 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  ALUP11 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  ALUP3 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  ALUP4 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  AMAR3 — 259 sem | corte: 2024-04-05 | previsões OOS: 103
  AMBP3 — 259 sem | corte: 2024-04-05 | previsões OOS: 1

## MODELO 2: GARCH-X(1,1)
$(σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1} + γ·ΔSVI_{t-1})$

Janela móvel de tamanho fixo: 52 * 3 semanas

---
O que muda em relação ao Modelo 1:

A única diferença estrutural é o exógeno $\gamma \cdot \Delta SVI_{t-1}$, que aparece em dois momentos distintos:

**Na estimação** — `x=svi_treino` passa os 156 valores de $\Delta SVI$ da janela para a biblioteca `arch` estimar o parâmetro $\gamma$ junto com $(\omega, \alpha, \beta)$.

**Na previsão** — `x=svi_previsao` passa apenas o valor $\Delta SVI_{t-1}$, que é o único ponto do exógeno disponível no momento em que a previsão é feita. Isso garante que não há *look-ahead bias* — na semana $t$ você só conhece o SVI até a semana $t-1$.
```
Janela de treino:   svi_diff[ t-156 … t-1 ]  →  estima γ
Previsão:           svi_diff[ t-1 ]           →  usa γ para prever σ²_t


In [ ]:
resultados_m2 = []
log_m2        = []

for ticker, grupo in df_final.groupby("ticker_b3"):

    # ── 1. Prepara a série do ticker ──────────────────────────
    df_t = (
        grupo
        .sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv", "preco_abertura", "preco_fechamento", "svi_diff"]]
        .copy()
    )

    # Retorno log semanal com média removida
    df_t["ret"] = np.log(df_t["preco_fechamento"] / df_t["preco_abertura"])
    df_t["ret"] = df_t["ret"] - df_t["ret"].mean()
    df_t = df_t.dropna(subset=["ret"]).reset_index(drop=True)

    n = len(df_t)
    if n < JANELA_MINIMA + 1:
        print(f"  [PULADO] {ticker}: {n} semanas (mínimo {JANELA_MINIMA + 1})")
        log_m1.append({"ticker_b3": ticker, "status": "série curta", "previsoes": 0})
        continue

    semana_corte = df_t["semana"].iloc[JANELA_MINIMA]
    print(f"  {ticker} — {n} sem | corte: {semana_corte.date()} "
          f"| previsões OOS: {n - JANELA_MINIMA}")

    previsoes = []

    n = len(df_t)
    if n < JANELA_MINIMA + 1:
        print(f"  [PULADO] {ticker}: {n} semanas (mínimo {JANELA_MINIMA + 1})")
        log_m2.append({"ticker_b3": ticker, "status": "série curta", "previsoes": 0})
        continue

    semana_corte = df_t["semana"].iloc[JANELA_MINIMA]
    print(f"  {ticker} — {n} sem | corte: {semana_corte.date()} "
          f"| previsões OOS: {n - JANELA_MINIMA}")

    previsoes = []

    # ── 2. Rolling window (janela fixa) ───────────────────────
    for t in range(JANELA_MINIMA, n):

        retornos_treino = df_t["ret"].iloc[t - JANELA_MINIMA : t].values * 100

        # ΔSVI_{t-1}: janela de treino do exógeno
        # A janela vai de [t-JANELA_MINIMA : t], onde o último valor
        # é svi_diff_{t-1} — disponível no momento da previsão
        svi_treino = df_t["svi_diff"].iloc[t - JANELA_MINIMA : t].values.reshape(-1, 1)

        # ΔSVI_{t-1}: valor usado na equação de previsão
        # É o último valor da janela de treino (índice t-1)
        svi_previsao = df_t["svi_diff"].iloc[[t - 1]].values.reshape(-1, 1)

        # Pula semana se houver NaN no exógeno (não quebra o loop)
        if np.isnan(svi_treino).any() or np.isnan(svi_previsao).any():
            previsoes.append({
                "semana":     df_t["semana"].iloc[t],
                "rv":         df_t["rv"].iloc[t],
                "rv_pred_m2": np.nan,
                "ticker_b3":  ticker,
                "omega": np.nan, "alpha": np.nan,
                "beta":  np.nan, "gamma": np.nan,
            })
            continue

        try:
            # ── 3. Estima o GARCH-X(1,1) na janela ───────────
            modelo = arch_model(
                retornos_treino,
                mean="Constant",
                vol="GARCH",
                p=1, q=1,
                dist="normal",
                x=svi_treino, # γ·ΔSVI_{t-1} na estimação
            )
            fit = modelo.fit(
                disp="off",
                show_warning=False,
            )

            # ── 4. Previsão 1 passo à frente ──────────────────
            forecast = fit.forecast(
                horizon=1,
                reindex=False,
                x=svi_previsao,       # ΔSVI_{t-1} conhecido no momento t
            )
            var_pred = forecast.variance.values[-1, 0] / 10_000

            previsoes.append({
                "semana":      df_t["semana"].iloc[t],
                "rv":          df_t["rv"].iloc[t],
                "rv_pred_m2":  var_pred,
                "ticker_b3":   ticker,
                "omega":       fit.params.get("omega",    np.nan),
                "alpha":       fit.params.get("alpha[1]", np.nan),
                "beta":        fit.params.get("beta[1]",  np.nan),
                "gamma":       fit.params.get("x1",       np.nan),
            })

        except Exception as e:
            print(f"    [ERRO t={t} {ticker}]: {e}")
            previsoes.append({
                "semana":     df_t["semana"].iloc[t],
                "rv":         df_t["rv"].iloc[t],
                "rv_pred_m2": np.nan,
                "ticker_b3":  ticker,
                "omega": np.nan, "alpha": np.nan,
                "beta":  np.nan, "gamma": np.nan,
            })

    df_ticker = pd.DataFrame(previsoes)
    df_ticker["split"] = np.where(
        df_ticker["semana"] >= semana_corte,
        "out-of-sample",
        "in-sample",
    )

    resultados_m2.append(df_ticker)
    log_m2.append({
        "ticker_b3":     ticker,
        "status":        "ok",
        "semana_corte":  semana_corte,
        "previsoes_oos": (df_ticker["split"] == "out-of-sample").sum(),
        "nan_pct":       round(df_ticker["rv_pred_m2"].isna().mean() * 100, 1),
    })


df_m2     = pd.concat(resultados_m2, ignore_index=True)
df_log_m2 = pd.DataFrame(log_m2)

df_m2.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/previsoes_m2.csv", index=False)
df_log_m2.to_csv("log_m2.csv",   index=False)

print(f"\n✅ {len(df_m2)} linhas → previsoes_m2.csv")
print(f"   in-sample:     {(df_m2['split'] == 'in-sample').sum()} observações")
print(f"   out-of-sample: {(df_m2['split'] == 'out-of-sample').sum()} observações")
print(f"✅ Log de estimação → log_m2.csv")

A saída de streaming foi truncada nas últimas 5000 linhas.
    [ERRO t=157 CEDO3]: GARCH.__init__() got an unexpected keyword argument 'x'
    [ERRO t=158 CEDO3]: GARCH.__init__() got an unexpected keyword argument 'x'
    [ERRO t=159 CEDO3]: GARCH.__init__() got an unexpected keyword argument 'x'
    [ERRO t=160 CEDO3]: GARCH.__init__() got an unexpected keyword argument 'x'
    [ERRO t=161 CEDO3]: GARCH.__init__() got an unexpected keyword argument 'x'
    [ERRO t=162 CEDO3]: GARCH.__init__() got an unexpected keyword argument 'x'
    [ERRO t=163 CEDO3]: GARCH.__init__() got an unexpected keyword argument 'x'
    [ERRO t=164 CEDO3]: GARCH.__init__() got an unexpected keyword argument 'x'
    [ERRO t=165 CEDO3]: GARCH.__init__() got an unexpected keyword argument 'x'
    [ERRO t=166 CEDO3]: GARCH.__init__() got an unexpected keyword argument 'x'
    [ERRO t=167 CEDO3]: GARCH.__init__() got an unexpected keyword argument 'x'
    [ERRO t=168 CEDO3]: GARCH.__init__() got an unexpected ke

KeyboardInterrupt: 

In [ ]:
print(arch.__version__)
help(GARCH.__init__)

8.0.0
Help on function __init__ in module arch.univariate.volatility:

__init__(self, p: int = 1, o: int = 0, q: int = 1, power: float = 2.0) -> None
    Initialize self.  See help(type(self)) for accurate signature.



# Avaliação de Previsão e Inferência (Modelos Aninhados)



In [ ]:
# ══════════════════════════════════════════════════════════════
# FASE 4 — AVALIAÇÃO DE PREVISÃO E INFERÊNCIA
# Loss function: QLIKE
# Inferência:    Model Confidence Set (Hansen et al., 2011)
# ══════════════════════════════════════════════════════════════
from arch.bootstrap import MCS
import warnings
warnings.filterwarnings("ignore")

NIVEL_MCS = 0.10   # nível de significância do MCS (padrão na literatura)

# ── 1. Consolida as previsões dos três modelos ────────────────
df_fase4 = (
    df_m0[["semana", "ticker_b3", "rv", "rv_pred_m0", "split"]]
    .merge(df_m1[["semana", "ticker_b3", "rv_pred_m1", "split"]],
           on=["semana", "ticker_b3"], suffixes=("", "_m1"))
    .merge(df_m2[["semana", "ticker_b3", "rv_pred_m2", "split"]],
           on=["semana", "ticker_b3"], suffixes=("", "_m2"))
)

# Mantém apenas o período out-of-sample para avaliação
df_oos = df_fase4[df_fase4["split"] == "out-of-sample"].copy()
df_oos = df_oos.dropna(subset=["rv_pred_m0", "rv_pred_m1", "rv_pred_m2"])

print(f"Observações OOS para avaliação: {len(df_oos)}")


# ── 2. Função de perda QLIKE ──────────────────────────────────
# L_t(σ̂², RV) = RV/σ̂² - ln(RV/σ̂²) - 1
# Recomendada para variâncias pois é robusta a erros de medição na RV
def qlike(rv, pred):
    """QLIKE — retorna NaN se previsão <= 0 para evitar log indefinido."""
    pred = np.where(pred <= 0, np.nan, pred)
    return rv / pred - np.log(rv / pred) - 1

def mse(rv, pred):
    """MSE — complementar ao QLIKE para diagnóstico."""
    return (rv - pred) ** 2


df_oos["qlike_m0"] = qlike(df_oos["rv"], df_oos["rv_pred_m0"])
df_oos["qlike_m1"] = qlike(df_oos["rv"], df_oos["rv_pred_m1"])
df_oos["qlike_m2"] = qlike(df_oos["rv"], df_oos["rv_pred_m2"])

df_oos["mse_m0"]   = mse(df_oos["rv"], df_oos["rv_pred_m0"])
df_oos["mse_m1"]   = mse(df_oos["rv"], df_oos["rv_pred_m1"])
df_oos["mse_m2"]   = mse(df_oos["rv"], df_oos["rv_pred_m2"])


# ── 3. Perda média por ticker ─────────────────────────────────
metricas_ticker = (
    df_oos
    .groupby("ticker_b3")
    .agg(
        qlike_m0 = ("qlike_m0", "mean"),
        qlike_m1 = ("qlike_m1", "mean"),
        qlike_m2 = ("qlike_m2", "mean"),
        mse_m0   = ("mse_m0",   "mean"),
        mse_m1   = ("mse_m1",   "mean"),
        mse_m2   = ("mse_m2",   "mean"),
        n_obs    = ("rv",        "count"),
    )
    .reset_index()
)

# Ranking por QLIKE (menor = melhor)
metricas_ticker["melhor_qlike"] = (
    metricas_ticker[["qlike_m0", "qlike_m1", "qlike_m2"]]
    .idxmin(axis=1)
    .str.replace("qlike_", "M")
    .str.upper()
)

print("\n── Perda média QLIKE por ticker ──────────────────────────")
print(metricas_ticker[["ticker_b3", "qlike_m0", "qlike_m1",
                         "qlike_m2", "melhor_qlike"]].to_string(index=False))


# ── 4. Perda média agregada (todos os tickers) ────────────────
print("\n── Perda média agregada ──────────────────────────────────")
resumo_agregado = pd.DataFrame({
    "modelo":     ["M0 — Random Walk", "M1 — GARCH(1,1)", "M2 — GARCH-X(1,1)"],
    "qlike_mean": [df_oos["qlike_m0"].mean(),
                   df_oos["qlike_m1"].mean(),
                   df_oos["qlike_m2"].mean()],
    "qlike_std":  [df_oos["qlike_m0"].std(),
                   df_oos["qlike_m1"].std(),
                   df_oos["qlike_m2"].std()],
    "mse_mean":   [df_oos["mse_m0"].mean(),
                   df_oos["mse_m1"].mean(),
                   df_oos["mse_m2"].mean()],
})
print(resumo_agregado.to_string(index=False))


# ══════════════════════════════════════════════════════════════
# 5. MODEL CONFIDENCE SET (Hansen, Lunde & Nason, 2011)
#    Aplicado sobre a matriz de perdas QLIKE agregada
#    Block bootstrap para respeitar a dependência temporal
# ══════════════════════════════════════════════════════════════
print("\n── Model Confidence Set ──────────────────────────────────")

# Matriz de perdas: linhas = semanas, colunas = modelos
df_losses = (
    df_oos
    .groupby("semana")[["qlike_m0", "qlike_m1", "qlike_m2"]]
    .mean()                        # média cross-section por semana
    .dropna()
    .rename(columns={
        "qlike_m0": "M0_RW",
        "qlike_m1": "M1_GARCH",
        "qlike_m2": "M2_GARCH-X",
    })
)

mcs = MCS(
    df_losses,
    size=NIVEL_MCS,                # α = 0.10 (padrão na literatura)
    block_size=4,                  # bloco de 4 semanas (~1 mês)
    reps=5_000,                    # repetições bootstrap
    seed=42,
)
mcs.compute()

df_mcs = mcs.pvalues.reset_index()
df_mcs.columns = ["modelo", "mcs_pvalor"]
df_mcs["no_mcs"] = df_mcs["mcs_pvalor"] >= NIVEL_MCS   # True = sobrevive ao MCS

print(df_mcs.to_string(index=False))
print(f"\nModelos no MCS (p ≥ {NIVEL_MCS}):")
for _, row in df_mcs[df_mcs["no_mcs"]].iterrows():
    print(f"  ✅ {row['modelo']}  (p = {row['mcs_pvalor']:.4f})")
for _, row in df_mcs[~df_mcs["no_mcs"]].iterrows():
    print(f"  ❌ {row['modelo']}  (p = {row['mcs_pvalor']:.4f}) — eliminado")


# ── 6. Salva resultados ───────────────────────────────────────
df_oos.to_csv("avaliacao_oos.csv",           index=False)
metricas_ticker.to_csv("metricas_ticker.csv", index=False)
resumo_agregado.to_csv("resumo_agregado.csv", index=False)
df_mcs.to_csv("resultado_mcs.csv",            index=False)

print("\n✅ avaliacao_oos.csv    → perdas OOS por semana e ticker")
print("✅ metricas_ticker.csv → QLIKE e MSE médios por ticker")
print("✅ resumo_agregado.csv → perda média agregada dos 3 modelos")
print("✅ resultado_mcs.csv   → p-valores e resultado do MCS")